AuthorityEffect: None.


## 2. Three-Layer Sovereignty

To prevent LLMs from bluffing their way past errors, the Symbol Game enforces an architectural separation called **Three-Layer Sovereignty**:

1. **Layer 1: Continuous Model Weights (The Engine)**
   - Matrix multiplications, embeddings, transformer attention layers.
   - Fast, broad, heuristic pattern matching across enormous corpora.
   - *Cannot be trusted to referee itself.*

2. **Layer 2: Decoding and Sampling (The Actuator)**
   - Temperature, top-$p$, beam search, token generation strategies.
   - Explores proposals, generates candidate reasoning paths.

3. **Layer 3: Discrete External Host (The Sovereign Referee)**
   - Operates outside the neural network using exact rational arithmetic ($\mathbb{Q}$).
   - Evaluates constraints, checks cycle invariants, calculates boundary defects $\Phi_1$.
   - **Fail-Closed**: If the candidate violates any constraint, Layer 3 immediately halts and refuses.
   - It cannot be persuaded, sweet-talked, or distracted by fluent token sequences.


In [ ]:
# [Pinned] Environment verification & basic dependencies
import sys
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path
from typing import List, Tuple

assert sys.version_info >= (3, 10), "Requires Python >= 3.10"
print(f"[OK] Python version: {sys.version.split()[0]}")

In [ ]:
import sys

_REPO = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "omega_engine/crates_rs/omega_functional").is_dir()
)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
from omega_engine_woland.infrastructure.providers.adapters.native_functional import compute_graph_phi1

# [Proved] The 3-Cycle Invariant: Summing differences around a closed loop
claimed_differences = {
    ("A", "B"): Fraction(1),
    ("B", "C"): Fraction(1),
    ("C", "A"): Fraction(1),
}

cycle = [("A", "B"), ("B", "C"), ("C", "A")]
cycle_sum = sum(claimed_differences[edge] for edge in cycle)

print(f"Cycle: {' -> '.join(e[0] for e in cycle)} -> {cycle[0][0]}")
print(f"Sum around cycle: {cycle_sum}")

# The host computes the cycle defect Phi_1
phi_1, _gauge = compute_graph_phi1(
    ["A", "B", "C"], [(u, v, 1) for u, v in cycle], claimed_differences
)
verdict = "CONTRADICTION" if phi_1 > 0 else "CONSISTENT"

print(f"Defect Phi_1: {phi_1}")
print(f"Host Verdict: {verdict}")

assert phi_1 == Fraction(3), f"Expected defect 3, got {phi_1}"
assert verdict == "CONTRADICTION", "Cycle must be recognized as contradictory"
print("[OK] 3-cycle defect computed and verified!")

In [ ]:
import sys
from pathlib import Path

_REPO = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "omega_engine/crates_rs/omega_functional").is_dir()
)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))
from omega_engine_woland.infrastructure.providers.adapters.native_functional import compute_graph_phi1

# [Declared] Implementation of a Minimal Layer-3 Constraint Host Referee


@dataclass(frozen=True)
class EdgeConstraint:
    source: str
    target: str
    difference: Fraction  # target - source


class MinimalHostReferee:
    """A minimal Layer-3 Host Referee for directed cycle consistency."""

    @staticmethod
    def evaluate_cycle(constraints: List[EdgeConstraint]) -> Tuple[Fraction, str]:
        """Calculates exact rational defect Phi_1 and returns a fail-closed verdict."""
        n = len(constraints)
        for i in range(n):
            curr_target = constraints[i].target
            next_source = constraints[(i + 1) % n].source
            if curr_target != next_source:
                raise ValueError(f"Disconnected cycle at {curr_target} != {next_source}")

        sum(c.difference for c in constraints)
        phi_1, _gauge = compute_graph_phi1(
            list(dict.fromkeys(c.source for c in constraints)),
            [(c.source, c.target, 1) for c in constraints],
            {(c.source, c.target): c.difference for c in constraints},
        )
        verdict = "CONSISTENT" if phi_1 == 0 else "CONTRADICTION"
        return phi_1, verdict


referee = MinimalHostReferee()

# Case A: The contradictory triangle from Section 1
triangle_bad = [
    EdgeConstraint("A", "B", Fraction(1)),
    EdgeConstraint("B", "C", Fraction(1)),
    EdgeConstraint("C", "A", Fraction(1)),
]
phi_bad, verdict_bad = referee.evaluate_cycle(triangle_bad)
print(f"Case A (Inconsistent): Defect={phi_bad}, Verdict={verdict_bad}")
assert verdict_bad == "CONTRADICTION"

# Case B: A consistent triangle (e.g. x_A=0, x_B=1, x_C=3 => diffs: +1, +2, -3)
triangle_good = [
    EdgeConstraint("A", "B", Fraction(1)),
    EdgeConstraint("B", "C", Fraction(2)),
    EdgeConstraint("C", "A", Fraction(-3)),
]
phi_good, verdict_good = referee.evaluate_cycle(triangle_good)
print(f"Case B (Consistent):   Defect={phi_good}, Verdict={verdict_good}")
assert verdict_good == "CONSISTENT"
print("[OK] Layer-3 host referee verified both consistent and inconsistent cases!")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `00_symbol_game_for_beginners`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
